# Cek Dataset

In [ ]:
from pathlib import Path
import zipfile
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
AWAL = ROOT / "data" / "Dataset DA Gelar Rasa 2026.zip"
AKHIR = ROOT / "data" / "Dataset GelarRasa Bismillah aja (3).xlsx"
if not AWAL.is_file() or not AKHIR.is_file():
    raise FileNotFoundError("Letakkan kedua dataset pada folder data/")
with zipfile.ZipFile(AWAL) as z:
    nama = z.namelist()
    assert sum(x.endswith("_AQMS.csv") for x in nama) == 27
    assert "HLT-01_health_outcomes_daily.csv" in nama
print("Dataset awal:", len(nama), "file | 27 AQMS")


## Cek sheet dan kolom

In [ ]:
butuh = {
    "01_AQMS_Harian": ["tanggal", "sensor_id", "polutan", "konsentrasi_ug_m3"],
    "03_AQMS_Stasiun": ["tanggal", "sensor_id", "pm25_ug_m3", "pm10_ug_m3"],
    "04_Meteo_Satuan": ["tanggal", "sensor_id", "temperature_c", "humidity_pct"],
    "05_Kesehatan_Harian": ["date", "hospital_id", "resp_admissions_under5", "resp_admissions_over60"],
    "06_Polusi_Kesehatan": ["prov_key", "kota_key", "tanggal", "pm25_ug_m3"],
    "07_Korelasi_Lag": ["polutan", "indikator", "lag_hari", "korelasi_pearson"],
    "08_Prioritiy_Area": ["provinsi", "kota_kabupaten", "status"],
    "10_ISPU_Harian": ["tanggal", "sensor_id", "ispu_max", "kategori_ispu"],
    "10_ISPU_Bulanan_Provinsi": ["bulan", "provinsi", "rata_rata_ispu_stasiun"],
}
from openpyxl import load_workbook
book = load_workbook(AKHIR, read_only=True, data_only=True)
assert set(butuh).issubset(book.sheetnames)
print("Sheet akhir:", len(book.sheetnames))
for sheet, cols in butuh.items():
    header = list(next(book[sheet].iter_rows(min_row=1, max_row=1, values_only=True)))
    hilang = sorted(set(cols)-set(header))
    assert not hilang, f"{sheet}: kolom tidak ada {hilang}"
    print("OK:", sheet)

book.close()


## Bandingkan ringkasan

In [ ]:
# Hitung baris pada tiga sheet utama tanpa memuat seluruh workbook.
import re
with zipfile.ZipFile(AKHIR) as z:
    for idx, sheet in [(3, "03_AQMS_Stasiun"), (36, "05_Kesehatan_Harian"), (41, "10_ISPU_Harian")]:
        with z.open(f"xl/worksheets/sheet{idx}.xml") as f:
            n_baris = len(re.findall(rb"<row\s+r=", f.read())) - 1
        print(sheet, "baris data:", n_baris)


## Bandingkan output cleaning

In [ ]:
hasil = ROOT / "hasil_cleaning_final"
if (hasil / "03_kesehatan_clean.csv").is_file():
    bersih = pd.read_csv(hasil / "03_kesehatan_clean.csv", usecols=["date", "hospital_id"])
    print("Kesehatan hasil cleaning awal:", len(bersih))
    print("Jumlah pada Excel akhir dapat berbeda karena tahap pengolahan lanjutan.")
else:
    print("Jalankan cleaning.ipynb untuk audit hasil cleaning awal.")
